In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
#from tqdm import tqdm
import re
import os
from tqdm.notebook import tqdm

#from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans,MiniBatchKMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

In [2]:
#os.getcwd()
#df = pd.read_excel('../data/raw/online_retail_II.xlsx')

In [3]:
#df.to_csv("../data/raw/online_retail.csv", index=False)
df = pd.read_csv("../data/raw/online_retail.csv")

In [4]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['status'] = pd.NA

negative_rows = df[df['Quantity'] < 0].sort_values('InvoiceDate')

for neg_i in tqdm(negative_rows.index, total=len(negative_rows)):
    
    neg_row = df.loc[neg_i]
    
    candidates = df[
        (df['Quantity'] > 0) &
        (df['status'].isna()) &
        (df['Customer ID'] == neg_row['Customer ID']) &
        (df['StockCode'] == neg_row['StockCode']) &
        (df['InvoiceDate'] <= neg_row['InvoiceDate']) &
        (df['Quantity'] == -neg_row['Quantity']) &
        (df['Price'] == neg_row['Price'])
    ]
    
    if len(candidates) > 0:
        match_i = candidates['InvoiceDate'].idxmax()
        df.loc[match_i, 'status'] = 'cancelled'
        df.loc[neg_i, 'status'] = 'return: purchase found'
    else:
        df.loc[neg_i, 'status'] = 'return: purchase not found'

df.loc[
    (df['Quantity'] > 0) & (df['status'].isna()),
    'status'
] = 'completed'

  0%|          | 0/12326 [00:00<?, ?it/s]

In [6]:
df.to_csv('large_online_retail.csv')

In [16]:
df['status'].value_counts()

status
completed                     510023
return: purchase not found      9214
cancelled                       3112
return: purchase found          3112
Name: count, dtype: int64

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  str           
 1   StockCode    525461 non-null  str           
 2   Description  522533 non-null  str           
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[us]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  str           
 8   status       525461 non-null  object        
dtypes: datetime64[us](1), float64(2), int64(1), object(1), str(4)
memory usage: 36.1+ MB


In [12]:
customer_id = df['Customer ID'].dropna()

fraction_rows = df[
    df['Customer ID'].notna() &
    (df['Customer ID'] % 1 != 0)
]

fraction_rows.shape[0]

0

In [14]:
df['Customer ID'] = df['Customer ID'].astype('Int64')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  str           
 1   StockCode    525461 non-null  str           
 2   Description  522533 non-null  str           
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[us]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  Int64         
 7   Country      525461 non-null  str           
 8   status       525461 non-null  object        
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), object(1), str(4)
memory usage: 36.6+ MB


In [15]:
neg = df[df['Quantity'] < 0]

starts_with_c = neg['Invoice'].astype(str).str.startswith('C').sum()

not_starts_with_c = (
    ~neg['Invoice'].astype(str).str.startswith('C')
).sum()

print("Negative quantity & Invoice starts with C:", starts_with_c)
print("Negative quantity & Invoice does NOT start with C:", not_starts_with_c)

Negative quantity & Invoice starts with C: 10205
Negative quantity & Invoice does NOT start with C: 2121


In [18]:
neg.to_csv('neg_Quant.csv')

In [100]:
neg = df[df['Quantity'] < 0].copy()

matched = neg.merge(
    df[df['Quantity'] > 0],
    on=['Customer ID', 'StockCode'],
    suffixes=('_return', '_purchase')
)

In [103]:
matched = matched[
    [
        'Invoice_return',
        'Invoice_purchase',
        'StockCode',
        'Description_return',
        'Description_purchase',
        'Quantity_return',
        'Quantity_purchase'
    ]
]

In [104]:
matched.to_csv("return.csv", index=False)

In [89]:
negative_df = df[
    (df['Quantity'] < 0) |
    (df['Price'] < 0)
].copy()

negative_df.to_csv(
    "negative_quantity_or_price.csv",
    index=False
)

negative_df.shape

(12329, 8)

In [84]:
df['StockCode'] = df['StockCode'].astype(str).str.upper().str.strip()
df['Description'] = df['Description'].astype(str).str.upper().str.strip()

In [85]:
collapsed = (
    df.groupby(['StockCode', 'Description'])
      .size()
      .rename('count')
      .reset_index()
)

collapsed['rn'] = (
    collapsed.groupby('StockCode')['count']
             .rank(method='first', ascending=False)
)

collapsed = collapsed[collapsed['rn'] == 1]

In [86]:
df = df.merge(collapsed, on='StockCode', how='left', suffixes=('', '_top'))
df['Description'] = df['Description'].fillna(df['Description_top'])
df = df.drop(columns='Description_top')

In [74]:
print('Unique entries')
df.nunique();

Unique entries


Invoice        28816
StockCode       4480
Description     4633
Quantity         825
InvoiceDate    25296
Price           1606
Customer ID     4383
Country           40
count            593
rn                 1
dtype: int64

In [75]:
print('Number of Null values')
df[['StockCode', 'Description']].isna().sum()

Number of Null values


StockCode        0
Description    363
dtype: int64

In [76]:
# 1) collapse first
collapsed0 = df.groupby(['StockCode', 'Description']).size().reset_index(name='count')

<class 'pandas.DataFrame'>
RangeIndex: 4871 entries, 0 to 4870
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   StockCode    4871 non-null   str  
 1   Description  4871 non-null   str  
 2   count        4871 non-null   int64
dtypes: int64(1), str(2)
memory usage: 114.3 KB


In [77]:
# 1) collapse first
collapsed = df.groupby(['StockCode', 'Description']).size().reset_index(name='count')

In [79]:
problem_df = (
    collapsed[collapsed.groupby('StockCode')['Description'].transform('nunique').gt(1)]
    .sort_values('StockCode')
)

problem_df.to_csv("problem_stockcode_description.csv", index=False)

In [80]:
problem_df2 = (
    collapsed[collapsed.groupby('Description')['StockCode'].transform('nunique').gt(1)]
    .sort_values('Description')
)

problem_df2.to_csv("problem_description_stockcode.csv", index=False)

In [81]:
len(problem_df), len(problem_df2)

(1390, 301)

In [90]:
df['Country'].value_counts();
df['line_cost']=df['Quantity']*df['Price']

In [91]:
# Group by customer and invoice, then sum the costs and quantities
invoice_df = df.groupby(['Customer ID', 'Invoice']).agg(
    total_cost=('line_cost', 'sum'),
    number_of_items=('Quantity', 'sum')
).reset_index()


invoice_df.to_csv("invoice.csv", index=False)

In [92]:
np.min(invoice_df['number_of_items'])

np.int64(-87167)

In [11]:
# Sort descending and take the head
invoice_df.sort_values(by='number_of_items', ascending=False).head(10);

NameError: name 'invoice_df' is not defined

In [10]:
# Sort ascending and take the head
bottom_10_invoices = invoice_df.sort_values(by='number_of_items', ascending=True).head(10)
bottom_10_invoices;

NameError: name 'invoice_df' is not defined

In [23]:
product_table = (
    df[['StockCode', 'Description']]
    .dropna()
    .drop_duplicates(subset=['StockCode'])
    .copy()
)

product_table['Description'] = product_table['Description'].astype(str)

print(product_table.shape)

(4632, 2)


In [24]:
def clean_product_description(text):
    text = str(text).lower()

    # remove numbers and measurements
    text = re.sub(r'\b\d+\s*(cm|mm|m|inch|inches|oz|ml|l|kg|g)\b', ' ', text)
    text = re.sub(r'\b\d+\b', ' ', text)

    # remove punctuation
    text = re.sub(r'[^a-z\s]', ' ', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # words that usually do not define category
    noise_words = {
        'set', 'pack', 'piece', 'pieces', 'small', 'large', 'medium',
        'mini', 'big', 'assorted', 'colour', 'colours', 'red', 'blue',
        'green', 'pink', 'white', 'black', 'yellow', 'purple', 'orange',
        'brown', 'silver', 'gold', 'metal', 'wooden', 'wood', 'glass',
        'ceramic', 'paper', 'plastic', 'felt', 'cotton', 'cm', 'mm',
        'round', 'square', 'heart', 'star', 'vintage', 'retro'
    }

    words = text.split()
    words = [w for w in words if w not in noise_words]

    return ' '.join(words)

In [25]:
product_table['clean_description'] = product_table['Description'].apply(
    clean_product_description
)

In [26]:
product_table.to_csv('product_table.csv')

In [27]:
(df['Quantity'] < 0).sum()

np.int64(12326)